## DATA PIPELINE


In [14]:
import time
from pathlib import Path
import requests
import re
import csv
from collections import Counter
import pandas as pd

#helper modules
from Python_code.modules import json_parser_theo as jp, classify_chats as cc, paser_chatti_html as hp

#pfade anlegen
repo = Path(".").resolve().parent
path_raw     = repo / "data/raw/data_sosci"
path_uploads = path_raw / "uploads"
csv_path     = path_raw / "daten.csv"
repo         = Path(".").resolve().parent
mapping_path = path_raw / "chatlog_mapping.csv"
uploads_root = path_raw / "uploads"
path_proc    = repo / "data/processed"

#laden der keys
path_sosci_key        = repo / "API_KEYS/sosci_data_key.txt"
path_sosci_per_header = repo / "API_KEYS/sosci_per_header.txt"

#laden der url inclusive key
sosci_data_url = path_sosci_key.read_text().strip()

# Header-Wert robust einlesen
raw_header = path_sosci_per_header.read_text().strip()
if raw_header.lower().startswith("authorization:"):
    raw_header = raw_header.split(":", 1)[1].strip()

headers = {"Authorization": raw_header}
BASE = "https://survey.ifkw.lmu.de/admin/api.php?v1"
uploads_list_url = f"{BASE}/projects/5393/uploads" #wenn man ein / am Ende einfügt erzeugt das ein fehler

#CSV API-Abfrage
response = requests.get(sosci_data_url)
response.raise_for_status()
csv_path.write_bytes(response.content)
print(f"CSV gespeichert unter: {csv_path}")


#uploads-api-abfrage
r = requests.get(uploads_list_url, headers=headers)
r.raise_for_status()
filenames = r.json()["files"]
print(f"{len(filenames)} hochgeladene Dateien gefunden.")

pattern = re.compile(r"^([A-Za-z]+\d+)\.(\d+)\.(\w+)$")
mapping_rows = []

for fname in filenames:
    match = pattern.match(fname)
    if not match:
        print(f"Dateiname passt nicht zum erwarteten Muster: {fname}")
        continue

    frage_code, teilnehmer_id, ext = match.groups()
    teilnehmer_id = str(int(teilnehmer_id))
    # Unterordner pro Person anlegen: uploads/<teilnehmer_id>/<fname>
    person_dir = path_uploads  / teilnehmer_id
    person_dir.mkdir(parents=True, exist_ok=True)
    out_path = person_dir / fname

    try:
        file_response = requests.get(f"{uploads_list_url}{"/"}{fname}", headers=headers)
        file_response.raise_for_status()
        out_path.write_bytes(file_response.content)
        print(f"Gespeichert: {out_path}")
        mapping_rows.append({"teilnehmer_id": teilnehmer_id, "frage_code": frage_code,
                              "dateiname": fname, "pfad": str(out_path)})
    except requests.exceptions.RequestException as e:
        print(f"Fehler bei {fname}: {e}")

    time.sleep(0.2) #sleep timer für den armen server

#mapping speicher, für weiterverarbeitung

mapping_path = path_raw / "chatlog_mapping.csv"
with open(mapping_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["teilnehmer_id", "frage_code", "dateiname", "pfad"])
    writer.writeheader()
    writer.writerows(mapping_rows)

print(f"\nZuordnungstabelle gespeichert unter: {mapping_path}")

#Kontrolle wievele vhats per person
counts = Counter(row["teilnehmer_id"] for row in mapping_rows)
print("\nChats pro Teilnehmer-ID:")
for tid, n in sorted(counts.items()):
    print(f"  {tid}: {n} Datei(en)")

CSV gespeichert unter: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/daten.csv
8 hochgeladene Dateien gefunden.
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/111/CH10.000111.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/112/CH10.000112.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/110/CH11.000110.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/111/CH11.000111.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/110/CH12.000110.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/111/CH12.000111.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/110/CH13.000110.html
Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sos

In [15]:

################
# Survey-CSV bereinigen (Zeit-/Metavariablen entfernen)
###############
df_raw = pd.read_csv(csv_path, sep="\t")

drop_exact = ["MAILSENT", "LASTDATA", "Q_VIEWER", "LASTPAGE", "MAXPAGE",
              "MODE", "STARTED", "REF", "QUESTNNR"]
time_cols  = [c for c in df_raw.columns if c.startswith("TIME")]   # TIME001..TIME_SUM
drop_cols  = [c for c in (drop_exact + time_cols) if c in df_raw.columns]

df_survey_clean = df_raw.drop(columns=drop_cols)
df_survey_clean.to_csv(path_proc / "survey_clean.csv", index=False)
print(f"Survey bereinigt: {df_raw.shape[1]} -> {df_survey_clean.shape[1]} Spalten "
      f"({len(drop_cols)} entfernt)")

###############################################################
# DATENSATZ 2: Chat-Nachrichten (JSON aus CSV + HTML-Uploads)#
################################################################

# Spalten mit JSON-Direkteingabe
JSON_INPUT_COLS = ["CH02s", "CH06s", "CH07s", "CH08s", "CH09s"]


# JSONs aus der CSV (Export-Prompt wird im Parser gefiltert) ----
rows_json = []
for _, r in df_raw.iterrows():
    tid = str(int(r["CASE"]))
    for col in JSON_INPUT_COLS:
        if col not in df_raw.columns or pd.isna(r[col]):
            continue
        messages = jp.extract_messages(str(r[col]))
        for turn, m in enumerate(messages, start=1):
            rows_json.append({
                "teilnehmer_id": tid,
                "frage_code":    col[:-1],          # CH02s -> CH02
                "quelle":        "json_csv",
                "turn":          turn,
                "role":          m["role"],
                "content":       m["content"],
            })
df_json = pd.DataFrame(rows_json)
#HTMLs aus den Uploads (via Mapping)
rows_html = []
mapping = pd.read_csv(mapping_path, dtype={"teilnehmer_id": str})
for _, r in mapping.iterrows():
    tid       = str(int(r["teilnehmer_id"]))
    html_path = uploads_root / tid / r["dateiname"]
    if not html_path.exists():
        print(f"HTML fehlt: {html_path}")
        continue
    for turn, m in enumerate(hp.extract(html_path), start=1):
        rows_html.append({
            "teilnehmer_id": tid,
            "frage_code":    r["frage_code"],
            "quelle":        "html_upload",
            "turn":          turn,
            "role":          m["speaker"],          # HTML-Parser: "speaker"/"text"
            "content":       m["text"],
        })
df_html = pd.DataFrame(rows_html)

# Zusammenführen + chat_id vergeben ----
df_chats = pd.concat([df_json, df_html], ignore_index=True)
df_chats["chat_uid"] = df_chats["teilnehmer_id"] + "_" + df_chats["frage_code"]
df_chats["chat_id"]  = df_chats.groupby("chat_uid").ngroup() + 1
df_chats = df_chats.sort_values(["chat_id", "turn"]).reset_index(drop=True)

df_chats.to_csv(path_proc / "chats_long.csv", index=False)
print(f"Chat-Nachrichten: {len(df_chats)} | eindeutige Chats: {df_chats['chat_id'].nunique()}")

Überspringe Turn 'conversation-turn-3' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-4' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-5' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-3' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-4' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-5' (nicht gerendert beim Export)


Survey bereinigt: 142 -> 92 Spalten (50 entfernt)


Überspringe Turn 'conversation-turn-1' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-2' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-3' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-4' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-5' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-6' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-7' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-8' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-9' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-10' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-11' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-12' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-13' (nicht gerendert beim Export)
Überspringe Turn 'conversation-turn-14' (nicht gerendert beim Export)
Überspringe Turn 'conversatio

Chat-Nachrichten: 68 | eindeutige Chats: 11


In [16]:
df_chats[df_chats["role"] == "user"]

,teilnehmer_id,frage_code,quelle,turn,role,content,chat_uid,chat_id
0,110,CH02,json_csv,1,user,ERROR: pip's dependency resolver does not curr...,110_CH02,1
2,110,CH06,json_csv,1,user,wie funktioniert das preismodell von langchain,110_CH06,2
4,110,CH06,json_csv,3,user,brauche ich einen api key von openai um die gb...,110_CH06,2
6,110,CH06,json_csv,5,user,hat google gemini kostenlose modelle für api a...,110_CH06,2
9,110,CH11,html_upload,2,user,GPT-5.5 hat erwartungsgemäß die besten Ergebni...,110_CH11,3
11,110,CH11,html_upload,4,user,ich will nur das du die rechtschreibung und we...,110_CH11,3
13,110,CH12,html_upload,1,user,ich habe eine datensatz mit vier varibalen cha...,110_CH12,4
15,110,CH12,html_upload,3,user,ich verwende python,110_CH12,4
17,110,CH12,html_upload,5,user,"so sieht mein code aus, def kippendorff_alpha(...",110_CH12,4
19,110,CH12,html_upload,7,user,"not_matched[""case_id""].to_list()",110_CH12,4


In [17]:
df_chats   = pd.read_csv(path_proc / "chats_long.csv")

df_labeled = cc.classify_chats(
    df_chats,
    model="gpt-5.5",
    api_key_path=repo / "API_KEYS/openai_key.txt",
)

df_labeled.to_csv(path_proc / "chats_labeled.csv", index=False)
df_labeled

    10/11 Chats klassifiziert ...
Fertig: 11 Chats klassifiziert.


,chat_id,teilnehmer_id,frage_code,task,sentiment,critical
0,1,110,CH02,Technische und analytische Unterstützung,Neutral,Nein
1,2,110,CH06,Informationssuche und Verständnis,Neutral,Nein
2,3,110,CH11,Schreiben und Textarbeit,Unfreundlich,Nein
3,4,110,CH12,Technische und analytische Unterstützung,Neutral,Nein
4,5,110,CH13,Technische und analytische Unterstützung,Neutral,Ja
5,6,111,CH02,Informationssuche und Verständnis,Neutral,Nein
6,7,111,CH10,Technische und analytische Unterstützung,Freundlich,Nein
7,8,111,CH11,Technische und analytische Unterstützung,Freundlich,Ja
8,9,111,CH12,Praktische Unterstützung und Strukturierung,Freundlich,Nein
9,10,111,CH13,Praktische Unterstützung und Strukturierung,Neutral,Ja
